# AutoFitter Tutorial

This notebook demonstrates how to use the `AutoFitter` class for automatic distribution selection and comparison.

## Overview

`AutoFitter` is designed to:
- Automatically test multiple probability distributions
- Select the best-fitting distribution based on various criteria (RMSE, AIC, BIC, etc.)
- Support all 113 SciPy continuous distributions
- Use lazy initialization for memory efficiency
- Provide comprehensive comparison tables and rankings

## Setup and Data Generation

Let's create some sample data that follows a known distribution to test AutoFitter's capability.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import magica as ma
from magica.core.auto_fitter import AutoFitter

# Set random seed for reproducibility
np.random.seed(123)

# Generate mixed wind speed data (complex scenario)
# Simulate different weather conditions
low_wind = np.random.weibull(2, 400) * 6 + 1    # Calm periods
normal_wind = np.random.lognormal(2, 0.5, 400)  # Variable periods  
high_wind = np.random.gamma(3, 2, 200)          # Storm periods

# Combine all periods
wind_data = np.concatenate([low_wind, normal_wind, high_wind])

print(f"Generated {len(wind_data)} wind speed measurements")
print(f"Min: {wind_data.min():.2f} m/s")
print(f"Max: {wind_data.max():.2f} m/s")
print(f"Mean: {wind_data.mean():.2f} m/s")
print(f"Std: {wind_data.std():.2f} m/s")

# Visualize the data
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.hist(wind_data, bins=50, density=True, alpha=0.7, color='skyblue', edgecolor='black')
plt.xlabel('Wind Speed (m/s)')
plt.ylabel('Density')
plt.title('Wind Speed Distribution')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(wind_data[:200], 'b-', alpha=0.7, linewidth=0.8)
plt.xlabel('Time')
plt.ylabel('Wind Speed (m/s)')
plt.title('Wind Speed Time Series (first 200 points)')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Basic AutoFitter Usage

Let's start with the default configuration that tests a curated set of stable distributions.

In [ ]:
# Load data into MagicA
processor = ma.read_data(wind_data)
print(f"Loaded data: {processor}")

# Create AutoFitter with default settings
auto_fitter = processor.get_auto_fitter(criterion='rmse')
print(f"\nCreated AutoFitter: {auto_fitter}")
print(f"Default candidates ({len(auto_fitter.candidates)}): {auto_fitter.candidates}")

In [ ]:
# Find the best distribution automatically
print("Finding best distribution from default candidates...")
best_result = auto_fitter.fit_best_distribution()

print(f"\n🏆 Best Distribution: {best_result['distribution']}")
print(f"📊 RMSE: {best_result['rmse']:.6f}")
print(f"📈 AIC: {best_result['aic']:.2f}")
print(f"📉 BIC: {best_result['bic']:.2f}")
print(f"🔍 KS p-value: {best_result['ks_pvalue']:.6f}")
print(f"⚙️ Parameters: {best_result['parameters']}")

## Comprehensive Distribution Comparison

Let's look at how all tested distributions performed.

In [ ]:
# Get comprehensive comparison table
comparison = auto_fitter.get_comparison_table(sort_by='rmse')

print("📊 Distribution Ranking (by RMSE):")
print("=" * 80)
print(f"{'Rank':<4} {'Distribution':<15} {'RMSE':<12} {'AIC':<10} {'KS p-value':<12} {'Status'}")
print("=" * 80)

successful_results = [(dist, result) for dist, result in comparison.items() if result['success']]
failed_results = [(dist, result) for dist, result in comparison.items() if not result['success']]

# Show top 10 successful fits
for i, (dist, result) in enumerate(successful_results[:10], 1):
    status = "✅ Good" if result['ks_pvalue'] > 0.05 else "⚠️ Fair"
    print(f"{i:<4} {dist:<15} {result['rmse']:<12.6f} {result['aic']:<10.1f} {result['ks_pvalue']:<12.6f} {status}")

print(f"\n📊 Summary:")
print(f"✅ Successful fits: {len(successful_results)}")
print(f"❌ Failed fits: {len(failed_results)}")
if failed_results:
    print(f"Failed distributions: {[dist for dist, _ in failed_results]}")

## Using the Best-Fitted Distribution

Once we have the best distribution, we can use it just like a regular MagicAdjuster.

In [ ]:
# Get the adjuster for the best distribution
best_adjuster = auto_fitter.get_best_adjuster()
print(f"Best adjuster: {best_adjuster}")

# Use it like any MagicAdjuster - calculate key statistics
mean_wind = best_adjuster.stats(moments='m')
percentile_50 = best_adjuster.ppf(0.5)   # Median
percentile_95 = best_adjuster.ppf(0.95)  # 95th percentile
percentile_99 = best_adjuster.ppf(0.99)  # 99th percentile

print(f"\n📊 Wind Speed Statistics (from best-fitted {best_result['distribution']}):")
print(f"Mean: {mean_wind:.2f} m/s")
print(f"Median (50th percentile): {percentile_50:.2f} m/s")
print(f"95th percentile: {percentile_95:.2f} m/s")
print(f"99th percentile: {percentile_99:.2f} m/s")

# Calculate probabilities for specific thresholds
prob_exceed_15 = 1 - best_adjuster.cdf(15)  # P(wind > 15 m/s)
prob_below_5 = best_adjuster.cdf(5)          # P(wind ≤ 5 m/s)

print(f"\n⚡ Risk Assessment:")
print(f"Probability of exceeding 15 m/s: {prob_exceed_15:.4f} ({prob_exceed_15*100:.2f}%)")
print(f"Probability of calm conditions (≤ 5 m/s): {prob_below_5:.4f} ({prob_below_5*100:.2f}%)")

## Testing All Available Distributions

For the most comprehensive analysis, let's test ALL 113 available distributions!

In [ ]:
# Get all available distributions
all_distributions = AutoFitter.get_all_available_distributions()
print(f"🌍 Total available distributions: {len(all_distributions)}")
print(f"First 20: {all_distributions[:20]}")
print(f"Last 20: {all_distributions[-20:]}")

# Create AutoFitter with ALL distributions
print("\n⚠️  Warning: Testing all distributions takes longer but is more comprehensive")
auto_fitter_comprehensive = processor.get_auto_fitter(
    candidates=all_distributions,  # Use ALL 113 distributions!
    criterion='aic'  # Let's use AIC this time
)

print(f"Created comprehensive AutoFitter: {auto_fitter_comprehensive}")

In [ ]:
# Find best from ALL distributions (this takes a moment...)
print("🔍 Testing ALL 113 distributions... (this may take 30-60 seconds)")
best_comprehensive = auto_fitter_comprehensive.fit_best_distribution()

print(f"\n🏆 Best Distribution (from ALL 113): {best_comprehensive['distribution']}")
print(f"📊 AIC: {best_comprehensive['aic']:.2f}")
print(f"📈 RMSE: {best_comprehensive['rmse']:.6f}")
print(f"🔍 KS p-value: {best_comprehensive['ks_pvalue']:.6f}")

# Compare with default selection
print(f"\n🔬 Comparison:")
print(f"Default selection ({len(auto_fitter.candidates)} dists): {best_result['distribution']} (RMSE: {best_result['rmse']:.6f})")
print(f"Comprehensive ({len(all_distributions)} dists): {best_comprehensive['distribution']} (RMSE: {best_comprehensive['rmse']:.6f})")

improvement = ((best_result['rmse'] - best_comprehensive['rmse']) / best_result['rmse']) * 100
print(f"Improvement: {improvement:.2f}% reduction in RMSE")

## Top Performers Analysis

Let's analyze the top-performing distributions from the comprehensive test.

In [ ]:
# Get comprehensive comparison
comprehensive_comparison = auto_fitter_comprehensive.get_comparison_table(sort_by='aic')
successful_comprehensive = [(dist, result) for dist, result in comprehensive_comparison.items() if result['success']]
failed_comprehensive = [dist for dist, result in comprehensive_comparison.items() if not result['success']]

print(f"📊 Comprehensive Results Summary:")
print(f"✅ Successfully fitted: {len(successful_comprehensive)}/113 distributions")
print(f"❌ Failed to fit: {len(failed_comprehensive)} distributions")
print(f"Success rate: {len(successful_comprehensive)/113*100:.1f}%")

print(f"\n🏆 Top 15 Distributions (by AIC):")
print("=" * 85)
print(f"{'Rank':<4} {'Distribution':<20} {'AIC':<10} {'RMSE':<12} {'KS p-val':<10} {'Fit Quality'}")
print("=" * 85)

for i, (dist, result) in enumerate(successful_comprehensive[:15], 1):
    quality = "Excellent" if result['ks_pvalue'] > 0.1 else "Good" if result['ks_pvalue'] > 0.05 else "Fair"
    emoji = "🥇" if i == 1 else "🥈" if i == 2 else "🥉" if i == 3 else "  "
    print(f"{emoji}{i:<2} {dist:<20} {result['aic']:<10.1f} {result['rmse']:<12.6f} {result['ks_pvalue']:<10.4f} {quality}")

if failed_comprehensive:
    print(f"\n❌ Failed distributions (first 10): {failed_comprehensive[:10]}")

## Custom Distribution Selection

For specific applications, you might want to test only relevant distributions.

In [ ]:
# Define wind-specific distributions
wind_specific_distributions = [
    'weibull_min',      # Most common for wind
    'weibull_max',      # Alternative Weibull
    'rayleigh',         # Theoretical wind model
    'lognorm',          # Common for environmental data
    'gamma',            # Flexible shape
    'chi2',             # Similar to gamma
    'maxwell',          # Physical distribution
    'rice',             # For wind with persistent component
    'gumbel_r',         # For extreme values
    'genextreme'        # Generalized extreme value
]

# Test wind-specific distributions
auto_fitter_wind = processor.get_auto_fitter(
    candidates=wind_specific_distributions,
    criterion='ks_pvalue'  # Use KS test p-value as criterion
)

print(f"🌪️  Testing wind-specific distributions: {wind_specific_distributions}")
best_wind = auto_fitter_wind.fit_best_distribution()

print(f"\n🏆 Best wind-specific distribution: {best_wind['distribution']}")
print(f"🔍 KS p-value: {best_wind['ks_pvalue']:.6f}")
print(f"📊 RMSE: {best_wind['rmse']:.6f}")

# Show wind-specific ranking
wind_comparison = auto_fitter_wind.get_comparison_table(sort_by='ks_pvalue')
print(f"\n🌪️  Wind-Specific Ranking (by KS p-value):")
print("-" * 60)
for i, (dist, result) in enumerate(wind_comparison.items(), 1):
    if result['success']:
        status = "✅" if result['ks_pvalue'] > 0.05 else "⚠️"
        print(f"{i:2}. {status} {dist:<15} p-value: {result['ks_pvalue']:.6f}")

## Different Selection Criteria Comparison

Different criteria can lead to different "best" distributions. Let's compare them.

In [ ]:
# Test the same distributions with different criteria
criteria = ['rmse', 'aic', 'bic', 'ks_pvalue', 'chi2_pvalue']
criteria_results = {}

for criterion in criteria:
    fitter = processor.get_auto_fitter(
        candidates=wind_specific_distributions,
        criterion=criterion
    )
    # Use already computed results if available
    if hasattr(auto_fitter_wind, '_results') and auto_fitter_wind._comparison_complete:
        fitter._results = auto_fitter_wind._results.copy()
        fitter._comparison_complete = True
    
    best = fitter.fit_best_distribution()
    criteria_results[criterion] = {
        'distribution': best['distribution'],
        'value': best[criterion] if criterion in best else best.get(criterion.replace('_', ''), 'N/A')
    }

print("📊 Best Distribution by Different Criteria:")
print("=" * 50)
print(f"{'Criterion':<15} {'Best Distribution':<15} {'Value'}")
print("=" * 50)

for criterion, result in criteria_results.items():
    value_str = f"{result['value']:.4f}" if isinstance(result['value'], (int, float)) else str(result['value'])
    print(f"{criterion:<15} {result['distribution']:<15} {value_str}")

# Count frequency of each distribution
dist_frequency = {}
for result in criteria_results.values():
    dist = result['distribution']
    dist_frequency[dist] = dist_frequency.get(dist, 0) + 1

print(f"\n🏆 Most Consistently Best Distribution:")
most_frequent = max(dist_frequency, key=dist_frequency.get)
print(f"{most_frequent} (selected by {dist_frequency[most_frequent]}/{len(criteria)} criteria)")

## Visualization: Comparing Top Distributions

Let's visually compare the top 3 distributions.

In [ ]:
# Get top 3 distributions from comprehensive analysis
top_3_dists = [dist for dist, _ in successful_comprehensive[:3]]
colors = ['red', 'blue', 'green']
linestyles = ['-', '--', '-.']

print(f"Visualizing top 3 distributions: {top_3_dists}")

# Create comparison plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Plot 1: Histogram with fitted PDFs
ax1.hist(wind_data, bins=40, density=True, alpha=0.6, color='lightgray', 
         label='Observed Data', edgecolor='black', linewidth=0.5)

# Generate smooth curves for top distributions
x_smooth = np.linspace(wind_data.min(), wind_data.max(), 500)

for i, dist_name in enumerate(top_3_dists):
    # Get the fitted adjuster for this distribution
    result = comprehensive_comparison[dist_name]
    
    # Create temporary processor and fit
    temp_processor = ma.read_data(wind_data)
    temp_processor.fit_distribution(dist_name)
    
    # Plot PDF
    pdf_smooth = temp_processor.pdf(x_smooth)
    ax1.plot(x_smooth, pdf_smooth, color=colors[i], linestyle=linestyles[i], 
            linewidth=2, label=f'{dist_name} (AIC: {result["aic"]:.1f})')

ax1.set_xlabel('Wind Speed (m/s)')
ax1.set_ylabel('Probability Density')
ax1.set_title('Top 3 Distributions - PDF Comparison')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Q-Q plot for best distribution
best_dist_name = top_3_dists[0]
temp_processor = ma.read_data(wind_data)
temp_processor.fit_distribution(best_dist_name)

# Generate theoretical quantiles
sorted_data = np.sort(wind_data)
n = len(sorted_data)
theoretical_quantiles = temp_processor.ppf(np.arange(1, n+1) / (n+1))

ax2.scatter(theoretical_quantiles, sorted_data, alpha=0.6, s=10, color='blue')
# Add reference line
min_val = min(theoretical_quantiles.min(), sorted_data.min())
max_val = max(theoretical_quantiles.max(), sorted_data.max())
ax2.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Fit')

ax2.set_xlabel(f'Theoretical Quantiles ({best_dist_name})')
ax2.set_ylabel('Observed Quantiles')
ax2.set_title(f'Q-Q Plot - {best_dist_name}')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print goodness-of-fit metrics for top 3
print("\n📊 Detailed Comparison of Top 3 Distributions:")
print("=" * 70)
print(f"{'Distribution':<20} {'AIC':<8} {'RMSE':<12} {'KS p-val':<10} {'Parameters'}")
print("=" * 70)

for dist_name in top_3_dists:
    result = comprehensive_comparison[dist_name]
    params_str = f"({len(result['parameters'])} params)"
    print(f"{dist_name:<20} {result['aic']:<8.1f} {result['rmse']:<12.6f} {result['ks_pvalue']:<10.4f} {params_str}")

## Summary and Best Practices

### Key AutoFitter Features:

1. **Multiple Testing Strategies**:
   - **Default**: 16 curated, stable distributions (fast, reliable)
   - **Comprehensive**: All 113 SciPy distributions (thorough, slower)
   - **Custom**: Domain-specific distribution subsets

2. **Selection Criteria**:
   - `rmse`: Root Mean Square Error (good for overall fit)
   - `aic`/`bic`: Information criteria (balance fit vs complexity)
   - `ks_pvalue`/`chi2_pvalue`: Statistical test p-values

3. **Memory Efficiency**:
   - Lazy initialization - distributions fitted only when tested
   - Failed fits don't crash the process
   - Comprehensive error handling

### Recommended Workflow:

```python
# 1. Start with default (fast screening)
auto_fitter = processor.get_auto_fitter()
best_default = auto_fitter.fit_best_distribution()

# 2. If needed, do comprehensive search
all_dists = AutoFitter.get_all_available_distributions()
auto_fitter_all = processor.get_auto_fitter(candidates=all_dists)
best_comprehensive = auto_fitter_all.fit_best_distribution()

# 3. Use the best distribution
best_adjuster = auto_fitter.get_best_adjuster()
percentile_95 = best_adjuster.ppf(0.95)
```

### When to Use What:

- **Default AutoFitter**: Quick analysis, most real-world cases
- **Comprehensive AutoFitter**: Research, when you need the absolute best fit
- **Custom candidates**: Domain expertise, specific requirements
- **Different criteria**: AIC/BIC for model selection, p-values for statistical validity

**AutoFitter makes distribution selection effortless while giving you complete control when needed!** 🎯